In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter

%matplotlib inline

# ADAPT THIS TO THE FILEPATH OF YOUR PARQUET FILE #
df = pd.read_parquet('nlp.parquet')

TOKENS_PER_SAMPLE = 1024
SMOOTHING_WINDOW = 64  # steps; scaled up proportionally for smaller batches below
BASE_BATCH_SIZE = 1024
COLORS = ['#0077BB', '#EE7733', '#009988']

# (width, batch_size, peak_lr, label) — three LR multipliers at fixed model size
configs = [
    (512, 256, 0.0009765625, 'LR ×1'),
    (512, 256, 0.00390625,   'LR ×8'),
    (512, 256, 0.015625,     'LR ×16'),
]

def smooth(values, window):
    if window <= 1:
        return values
    conv = np.convolve(values, np.ones(window) / window, mode='valid')
    # prepend cumulative means so output length matches input
    pad = [np.mean(values[:i+1]) for i in range(len(values) - len(conv))]
    return np.concatenate([pad, conv])

mpl.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 10, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 8, 'lines.linewidth': 1.2,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linewidth': 0.5,
    'pdf.fonttype': 42, 'ps.fonttype': 42,  # embed fonts for camera-ready PDFs
})

COL_WIDTH = 3.25
fig, ax = plt.subplots(figsize=(COL_WIDTH, COL_WIDTH * 0.8))

for i, (width, bs, lr, label) in enumerate(configs):
    run = df[(df['width'] == width) & (df['batch_size'] == bs) &
             (df['peak_lr'] == lr) & (df['seed'] == 111)
            ].sort_values('samples_seen').copy()

    # scale smoothing window to match effective optimizer steps vs. base batch size
    window = int(BASE_BATCH_SIZE / bs) * SMOOTHING_WINDOW if bs < BASE_BATCH_SIZE else SMOOTHING_WINDOW
    gns_smoothed = smooth(run['gns'].values, window)
    # convert samples → tokens for the x-axis
    tokens_seen = run['samples_seen'].values * TOKENS_PER_SAMPLE

    # GNS is in sample units; multiply by tokens-per-sample to get token units
    ax.plot(tokens_seen, gns_smoothed * TOKENS_PER_SAMPLE, color=COLORS[i], label=label)

ax.set_xlabel('# Tokens Seen')
ax.set_ylabel(r'GNS ($\times\!10^5$ Tokens)')
ax.set_title('GNS Dependency on Learning Rate', fontweight='medium', fontsize=10, pad=4)
ax.set_xlim(0, 1e10)
ax.set_ylim(0, 200000)

# manual ticks with a shared multiplier label instead of scientific notation per tick
ax.xaxis.set_major_locator(FixedLocator([0, 2e9, 4e9, 6e9, 8e9, 10e9]))
ax.xaxis.set_major_formatter(FixedFormatter(['0', '2', '4', '6', '8', '10']))
ax.text(1.0, -0.18, r'$\times 10^9$', transform=ax.transAxes, fontsize=9, ha='right', va='top')

ax.yaxis.set_major_locator(FixedLocator([0, 0.5e5, 1.0e5, 1.5e5, 2.0e5]))
ax.yaxis.set_major_formatter(FixedFormatter(['0', '0.5', '1.0', '1.5', '2.0']))

ax.legend(loc='best', fontsize=6, handlelength=1.5, handletextpad=0.4, frameon=True, framealpha=0.9)

plt.tight_layout()
fig.savefig('nlp_gns_dependency.pdf', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()
print('Saved nlp_gns_dependency.pdf')